# PSA Bronze Ingestion

This notebook ingests raw Philippine Statistics Authority (PSA) datasets from Google Drive into Delta tables in the Bronze layer.

The Bronze layer preserves source data with minimal transformation and adds ingestion metadata for lineage and traceability.

Business-level cleaning, geographic standardization, hierarchy resolution, joins, and analytical transformations will be performed later in the Silver layer.

In [0]:
from pyspark.sql import functions as F
import uuid
from datetime import datetime

# ── Google Drive connection (existing Unity Catalog connection) ──
GOOGLE_DRIVE_CONNECTION = "capstone_gdrive"

# ── Unity Catalog targets ────────────────────────────────────────
CATALOG = "tuklas_dev"
BRONZE_SCHEMA = "02_bronze"
INGESTION_SCHEMA = "01_ingestion"

# ── Ingestion run identifier (unique per notebook execution) ───────
run_id = f"psa_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# ── PSA source folder ────────────────────────────────────────────
PSA_FOLDER = "https://drive.google.com/drive/folders/1hBSj2dHu1ylbEjRjLzzxwMQ9CBXItiQs"

# ── Source file URLs and standardized filenames ──────────────────────
# File IDs are stable; filenames were standardized in Google Drive.
SOURCE_FILES = {
    "DS_01":      {"url": "https://drive.google.com/file/d/1_9OKDJpLbM-RaotcoiVz1EfNTDzTk1QK/view", "file": "DS_01_PSA_PTSA_Statistical_Tables_2000_2025.xlsx"},
    "DS_02":      {"url": "https://drive.google.com/file/d/1S2Q8P8zUJe84XHFMJgZT0QEkNSDdWnpe/view", "file": "DS_02_PSA_Tourism_Statistical_Classification_2025.xlsx"},
    "DS_08_17":   {"url": "https://drive.google.com/file/d/1CQlKwTsy62lLvVJZrTad0D2OvjZQ3FSz/view", "file": "DS_08_to_DS_17_PSA_PSY_Tables_8_1_to_8_10.xlsx"},
    "DS_18_xlsx": {"url": "https://drive.google.com/file/d/1PTQ2ahv-4_U9BZdnN92CKtCl9ZmBm33w/view", "file": "DS_18_PSA_Projected_Mid_Year_Population_2020_2025.xlsx"},
    "DS_18_pdf":  {"url": "https://drive.google.com/file/d/1yxMeiZWzEWjnG44knXsKNb0xHTyk3Ysg/view", "file": "DS_18_PSA_Projected_Mid_Year_Population_2020_2025.pdf"},
    "DS_26":      {"url": "https://drive.google.com/file/d/1Nyh5yZdeb7BvWCQFcS5k5gvTSyaC2m-u/view", "file": "DS_26_PSA_Table_3_63_Accommodation_Food_Service_GVA_2022_2024.csv"},
    "DS_27":      {"url": "https://drive.google.com/file/d/1juR4xXzxkwvhQECEzIhBSzbClmlxSSwR/view", "file": "DS_27_PSA_Table_3_72_Total_GDP_Current_Prices_2019_2023.csv"},
    "DS_28":      {"url": "https://drive.google.com/file/d/1p1Hy-0uqHi5Thanam71kSfgZApGgkJSA/view", "file": "DS_28_PSA_Table_3_73_Total_GDP_Constant_Prices_2019_2023.csv"},
    "DS_29":      {"url": "https://drive.google.com/file/d/1GwZU87TEmiv3smIZ5sqxJADc0ARsshuV/view", "file": "DS_29_PSA_Table_3_74_Per_Capita_GDP_Current_Prices_2019_2023.csv"},
    "DS_30":      {"url": "https://drive.google.com/file/d/12QzN6oZ-ly9WkqMUt0PmZNZSKBhP73RH/view", "file": "DS_30_PSA_Table_3_75_Per_Capita_GDP_Constant_Prices_2019_2023.csv"},
    "DS_31":      {"url": "https://drive.google.com/file/d/1aMLLBTcbH7WoRFrzs-ApRanMYxTRw-F-/view", "file": "DS_31_PSA_Table_3_94_Accommodation_Food_Service_GVA_Current_Prices_2019_2023.csv"},
    "DS_32":      {"url": "https://drive.google.com/file/d/1pBUq0mKtnadLLc2wNGm3RcgRpjRTHmbi/view", "file": "DS_32_PSA_Table_3_95_Accommodation_Food_Service_GVA_Constant_Prices_2019_2023.csv"},
}

# ── PSY T8 sheet to canonical DS ID mapping ──────────────────────────
PSY_SHEET_TO_DS = {"T8_1": "DS_08", "T8_2": "DS_09", "T8_3": "DS_10", "T8_4": "DS_11", "T8_5": "DS_12", "T8_6": "DS_13", "T8_7": "DS_14", "T8_8": "DS_15", "T8_9": "DS_16", "T8_10": "DS_17"}

# ── DS_01 sheet to Bronze table name mapping ───────────────────────
DS01_TABLE_MAP = {
    "Tables 1":  "psa_ds_01_table_01",
    "Tables 2":  "psa_ds_01_table_02",
    "Tables 3":  "psa_ds_01_table_03",
    "Tables 4":  "psa_ds_01_table_04",
    "Tables 5":  "psa_ds_01_table_05",
    "Tables 6":  "psa_ds_01_table_06",
    "Table 7":   "psa_ds_01_table_07",
    "Tables 8":  "psa_ds_01_table_08",
    "Tables 9":  "psa_ds_01_table_09",
    "Tables 10": "psa_ds_01_table_10",
}

# ── DS_02 sheet to Bronze table name mapping ───────────────────────
DS02_TABLE_MAP = {
    "Metadata":                "psa_ds_02_metadata",
    "2025 Tourism Industries": "psa_ds_02_tourism_industries",
    "2025 Tourism Products":   "psa_ds_02_tourism_products",
}

# ── Helper: fully qualified Bronze table name ─────────────────────
def bronze_table(table_name: str) -> str:
    """Return a fully qualified, backtick-quoted Bronze table identifier."""
    return f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`{table_name}`"

# ── Helper: fully qualified ingestion control table name ──────────
def ingestion_table(table_name: str) -> str:
    """Return a fully qualified, backtick-quoted ingestion control table identifier."""
    return f"`{CATALOG}`.`{INGESTION_SCHEMA}`.`{table_name}`"

# ── Reverse lookup: source_file -> source_path ──────────────────────
_FILE_TO_URL = {v["file"]: v["url"] for v in SOURCE_FILES.values()}

# ── Helper: add standard Bronze metadata columns ──────────────────
METADATA_COLS = ["_source_dataset_id", "_source_file", "_source_sheet", "_source_path", "_source_system", "_ingested_at"]

def add_bronze_metadata(df, dataset_id, source_file, source_path, source_sheet=None):
    """Add the six standard Bronze metadata columns to a DataFrame."""
    return (df
        .withColumn("_source_dataset_id", F.lit(dataset_id))
        .withColumn("_source_file",        F.lit(source_file))
        .withColumn("_source_sheet",        F.lit(source_sheet))
        .withColumn("_source_path",        F.lit(source_path))
        .withColumn("_source_system",      F.lit("PSA"))
        .withColumn("_ingested_at",        F.current_timestamp()))

def validate_bronze_table(tbl_name, expected_dataset_id=None, expected_file=None, expect_sheet=False):
    """Read back a Bronze table, print stats, verify metadata columns and values."""
    table_df = spark.table(bronze_table(tbl_name))
    row_count = table_df.count()
    col_count = len(table_df.columns)
    print(f"  Table  : {bronze_table(tbl_name)}")
    print(f"  Rows   : {row_count}")
    print(f"  Columns: {col_count}")
    table_df.printSchema()
    for col in METADATA_COLS:
        assert col in table_df.columns, f"Missing metadata column: {col}"
    # Verify _source_system = PSA
    assert table_df.filter(F.col("_source_system") != "PSA").count() == 0, f"_source_system not PSA in {tbl_name}"
    # Verify _source_path is not null
    assert table_df.filter(F.col("_source_path").isNull()).count() == 0, f"_source_path has nulls in {tbl_name}"
    # Verify _ingested_at is not null
    assert table_df.filter(F.col("_ingested_at").isNull()).count() == 0, f"_ingested_at has nulls in {tbl_name}"
    # Verify _source_dataset_id matches expected
    if expected_dataset_id:
        assert table_df.filter(F.col("_source_dataset_id") != expected_dataset_id).count() == 0, f"_source_dataset_id mismatch in {tbl_name}"
    # Verify _source_file matches expected
    if expected_file:
        assert table_df.filter(F.col("_source_file") != expected_file).count() == 0, f"_source_file mismatch in {tbl_name}"
    # Verify _source_sheet: populated for Excel, null for CSV/PDF
    if expect_sheet:
        assert table_df.filter(F.col("_source_sheet").isNull()).count() == 0, f"_source_sheet should be populated in {tbl_name}"
    else:
        assert table_df.filter(F.col("_source_sheet").isNotNull()).count() == 0, f"_source_sheet should be null in {tbl_name}"
    print(f"  Metadata verified: ds_id={expected_dataset_id or 'N/A'}, system=PSA, sheet={'populated' if expect_sheet else 'null'}.")
    return row_count, col_count

# ── Ingestion summary tracker ─────────────────────────────────────
ingestion_summary = []

def record_ingestion(source_dataset_id, source_file, source_sheet, bronze_tbl, row_count, col_count, status, notes="", source_path=None):
    """Append a row to the ingestion summary tracker AND persist to file_manifest via MERGE."""
    ingestion_summary.append({"source_dataset_id": source_dataset_id, "source_file": source_file, "source_sheet": source_sheet, "bronze_table": bronze_tbl, "row_count": row_count, "column_count": col_count, "status": status, "notes": notes})
    # Look up source_path if not provided
    if source_path is None:
        source_path = _FILE_TO_URL.get(source_file)
    # Build manifest row as a temp view, then MERGE INTO file_manifest
    manifest_row = spark.createDataFrame([(
        run_id, "01_ingest_psa", source_dataset_id, "PSA", source_file,
        source_sheet, source_path, bronze_tbl, row_count, col_count,
        status, notes,
    )], schema="run_id STRING, notebook_name STRING, source_dataset_id STRING, source_system STRING, source_file STRING, source_sheet STRING, source_path STRING, destination_table STRING, row_count BIGINT, column_count BIGINT, status STRING, notes STRING")
    manifest_row = (manifest_row
        .withColumn("discovered_at", F.current_timestamp())
        .withColumn("ingested_at", F.current_timestamp()))
    manifest_row.createOrReplaceTempView("_manifest_row")
    spark.sql(f"""
    MERGE INTO {ingestion_table('file_manifest')} AS target
    USING _manifest_row AS source
    ON target.run_id = source.run_id
       AND target.source_dataset_id = source.source_dataset_id
       AND target.source_file = source.source_file
       AND COALESCE(target.source_sheet, '') = COALESCE(source.source_sheet, '')
       AND target.destination_table = source.destination_table
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)

def log_ingestion_error(source_dataset_id, source_file, source_sheet, source_path, destination_table, error_type, error_message, notes=""):
    """Write an ingestion error to the ingestion_errors control table."""
    error_row = spark.createDataFrame([(
        run_id, "01_ingest_psa", source_dataset_id, "PSA", source_file,
        source_sheet, source_path, destination_table, error_type,
        error_message, None, notes,
    )], schema="run_id STRING, notebook_name STRING, source_dataset_id STRING, source_system STRING, source_file STRING, source_sheet STRING, source_path STRING, destination_table STRING, error_type STRING, error_message STRING, error_timestamp TIMESTAMP, notes STRING")
    error_row = error_row.withColumn("error_timestamp", F.current_timestamp())
    (error_row.write
        .format("delta")
        .mode("append")
        .saveAsTable(ingestion_table("ingestion_errors")))

print("Configuration loaded.")
print(f"  Catalog       : {CATALOG}")
print(f"  Bronze schema : {BRONZE_SCHEMA}")
print(f"  PSA folder    : {PSA_FOLDER}")
print(f"  GDrive conn   : {GOOGLE_DRIVE_CONNECTION}")
print(f"  Source files  : {len(SOURCE_FILES)} entries")
print(f"  Run ID        : {run_id}")

## Ingestion Control — Run Initialization

This cell inserts a `STARTED` record into `ingestion_run_log` before any PSA ingestion begins.

The `run_id` generated in the Configuration cell uniquely identifies this notebook execution. At the end of the notebook, this same row is updated to `SUCCESS`, `PARTIAL_SUCCESS`, or `FAILED` based on the results in `file_manifest`.

In [0]:
# ── Insert STARTED record into ingestion_run_log ──────────────────
run_started_at = datetime.now()

spark.sql(f"""
INSERT INTO {ingestion_table('ingestion_run_log')}
(run_id, notebook_name, source_system, source_folder, started_at, completed_at,
 files_discovered, files_ingested, files_skipped, files_failed, tables_created, status, notes)
VALUES ('{run_id}', '01_ingest_psa', 'PSA', '{PSA_FOLDER}', current_timestamp(), NULL,
        0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""")

print(f"✓ Run log STARTED record inserted")
print(f"  run_id  = {run_id}")
print(f"  started = {run_started_at.isoformat()}")

## Step 2 — Inspect Google Drive Folder

Before ingesting anything, we inspect the PSA folder through the existing Google Drive Unity Catalog connection (`capstone_gdrive`).

This step lists every file visible to Databricks — including file name, path, size, and modified timestamp — and confirms what the connector can access.

No files in Google Drive are modified, deleted, or created.

In [0]:
# List all files in the PSA Google Drive folder using the existing connection.
folder_listing = spark.sql(f"""
SELECT path, file, size, modification_time
FROM list_files(
  '{PSA_FOLDER}',
  connection => '{GOOGLE_DRIVE_CONNECTION}'
)
ORDER BY path
""")

print(f"Total items found: {folder_listing.count()}")
display(folder_listing)

## Step 3 — CSV Sources

The PSA folder contains seven CSV files, each a separate PSA dataset:

* **DS_26** — Table 3.63: Accommodation & Food Service GVA 2022-2024
* **DS_27** — Table 3.72: Total GDP at Current Prices 2019-2023
* **DS_28** — Table 3.73: Total GDP at Constant Prices 2019-2023
* **DS_29** — Table 3.74: Per Capita GDP at Current Prices 2019-2023
* **DS_30** — Table 3.75: Per Capita GDP at Constant Prices 2019-2023
* **DS_31** — Table 3.94: Accommodation & Food Service GVA at Current Prices 2019-2023
* **DS_32** — Table 3.95: Accommodation & Food Service GVA at Constant Prices 2019-2023

Each CSV is inspected individually because report structures may differ — title rows, header positions, footer rows, and delimiters vary across PSA publications.

### DS_32 — Accommodation and Food Service GVA

**Source file:** DS_32_PSA_Table_3_95_Accommodation_Food_Service_GVA_Constant_Prices_2019_2023.csv
**Bronze target:** `tuklas_dev`.`02_bronze`.`psa_ds_32`

The source CSV has a report title in row 1 and the real header in row 2, so we skip 1 row and treat the next row as the header.

Expected columns: `Provinces`, `2019`, `2020`, `2021`, `2022`, `2023`.

In [0]:
# ── DS_32: Read raw CSV from Google Drive ─────────────────────────
ds32_url = SOURCE_FILES["DS_32"]["url"]

ds32_df = (
    spark.read
    .format("csv")
    .option("databricks.connection", GOOGLE_DRIVE_CONNECTION)
    .option("skipRows", 1)        # Skip the TABLE 3.95 title row
    .option("header", "true")     # Row 2 becomes the real header
    .option("inferSchema", "true")
    .option("mode", "FAILFAST")
    .load(ds32_url)
)

# Inspect before writing
ds32_df.printSchema()
display(ds32_df.limit(20))

In [0]:
# ── DS_32: Add ingestion metadata and write to Bronze ─────────────
try:
    ds32_bronze = add_bronze_metadata(
        ds32_df,
        dataset_id="DS_32",
        source_file=SOURCE_FILES["DS_32"]["file"],
        source_path=SOURCE_FILES["DS_32"]["url"],
    )

    (ds32_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(bronze_table("psa_ds_32")))

    row_count, col_count = validate_bronze_table("psa_ds_32", "DS_32", SOURCE_FILES["DS_32"]["file"])
    display(spark.table(bronze_table("psa_ds_32")).limit(10))
    record_ingestion("DS_32", SOURCE_FILES["DS_32"]["file"], None,
        "psa_ds_32", row_count, col_count, "INGESTED")
except Exception as e:
    print(f"✗ FAILED: DS_32 — {e}")
    log_ingestion_error("DS_32", SOURCE_FILES["DS_32"]["file"], None,
        SOURCE_FILES["DS_32"]["url"], "psa_ds_32", "INGESTION_ERROR", str(e))
    record_ingestion("DS_32", SOURCE_FILES["DS_32"]["file"], None,
        "psa_ds_32", 0, 0, "FAILED", str(e))

### DS_30 — Inspection

**Source file:** DS_30_PSA_Table_3_75_Per_Capita_GDP_Constant_Prices_2019_2023.csv
**Bronze target:** `tuklas_dev`.`02_bronze`.`psa_ds_30`

Before ingesting, we read the raw CSV without any header assumptions to inspect the first rows and determine whether a descriptive title row precedes the actual header.

In [0]:
# ── DS_30: Inspect raw CSV structure ────────────────────────────────
ds30_url = SOURCE_FILES["DS_30"]["url"]

# Read raw (header=false) to see the actual first rows and identify the header position.
ds30_raw = (
    spark.read
    .format("csv")
    .option("databricks.connection", GOOGLE_DRIVE_CONNECTION)
    .option("header", "false")
    .load(ds30_url)
)

print("DS_30 — raw rows (first 5, single-column view):")
display(ds30_raw.limit(5))

# Findings:
#   Row 1: Title — "TABLE 3.75 Per Capita Gross Domestic Product, ..."
#   Row 2: Header — "Provinces","2019","2020","2021","2022","2023"
#   Row 3+: Data rows (quoted province names, numeric year columns)
# The title row's unquoted commas confuse the default parser.
# skipRows=1 is justified to reach the real header.
print("\n→ skipRows=1 justified: title in row 1, header in row 2.")

# Read properly with skipRows=1
ds30_df = (
    spark.read
    .format("csv")
    .option("databricks.connection", GOOGLE_DRIVE_CONNECTION)
    .option("skipRows", 1)
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mode", "FAILFAST")
    .load(ds30_url)
)

print("\nDS_30 — parsed schema:")
ds30_df.printSchema()
display(ds30_df.limit(20))

### DS_31 — Inspection

**Source file:** DS_31_PSA_Table_3_94_Accommodation_Food_Service_GVA_Current_Prices_2019_2023.csv
**Bronze target:** `tuklas_dev`.`02_bronze`.`psa_ds_31`

Same approach as DS_30 — read raw without header assumptions to identify the real header row.

In [0]:
# ── DS_31: Inspect raw CSV structure ────────────────────────────────
ds31_url = SOURCE_FILES["DS_31"]["url"]

# Read raw (header=false) to see the actual first rows.
ds31_raw = (
    spark.read
    .format("csv")
    .option("databricks.connection", GOOGLE_DRIVE_CONNECTION)
    .option("header", "false")
    .load(ds31_url)
)

print("DS_31 — raw rows (first 5, single-column view):")
display(ds31_raw.limit(5))

# Findings:
#   Row 1: Title — "TABLE 3.94 Gross Value Added in Accommodation and Food Service Activities, ..."
#   Row 2: Header — "Provinces","2019","2020","2021","2022","2023"
#   Row 3+: Data rows (quoted province names, numeric year columns)
# Same structure as DS_30 and DS_32 — skipRows=1 is justified.
print("\n→ skipRows=1 justified: title in row 1, header in row 2.")

# Read properly with skipRows=1
ds31_df = (
    spark.read
    .format("csv")
    .option("databricks.connection", GOOGLE_DRIVE_CONNECTION)
    .option("skipRows", 1)
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mode", "FAILFAST")
    .load(ds31_url)
)

print("\nDS_31 — parsed schema:")
ds31_df.printSchema()
display(ds31_df.limit(20))

### DS_30 — Write to Bronze and Validate

Now that inspection confirms the header is in row 2 (after the title), we write `ds30_df` (already parsed with `skipRows=1`) to the Bronze Delta table `tuklas_dev.02_bronze.psa_ds_30` with ingestion metadata.

In [0]:
# ── DS_30: Add ingestion metadata and write to Bronze ─────────────
try:
    ds30_bronze = add_bronze_metadata(
        ds30_df,
        dataset_id="DS_30",
        source_file=SOURCE_FILES["DS_30"]["file"],
        source_path=SOURCE_FILES["DS_30"]["url"],
    )

    (ds30_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(bronze_table("psa_ds_30")))

    row_count, col_count = validate_bronze_table("psa_ds_30", "DS_30", SOURCE_FILES["DS_30"]["file"])
    display(spark.table(bronze_table("psa_ds_30")).limit(10))
    record_ingestion("DS_30", SOURCE_FILES["DS_30"]["file"], None,
        "psa_ds_30", row_count, col_count, "INGESTED")
except Exception as e:
    print(f"✗ FAILED: DS_30 — {e}")
    log_ingestion_error("DS_30", SOURCE_FILES["DS_30"]["file"], None,
        SOURCE_FILES["DS_30"]["url"], "psa_ds_30", "INGESTION_ERROR", str(e))
    record_ingestion("DS_30", SOURCE_FILES["DS_30"]["file"], None,
        "psa_ds_30", 0, 0, "FAILED", str(e))

### DS_31 — Write to Bronze and Validate

Same pattern as DS_30 — `ds31_df` was already parsed with `skipRows=1` during inspection. We add metadata columns and write to `tuklas_dev.02_bronze.psa_ds_31`.

In [0]:
# ── DS_31: Add ingestion metadata and write to Bronze ─────────────
try:
    ds31_bronze = add_bronze_metadata(
        ds31_df,
        dataset_id="DS_31",
        source_file=SOURCE_FILES["DS_31"]["file"],
        source_path=SOURCE_FILES["DS_31"]["url"],
    )

    (ds31_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(bronze_table("psa_ds_31")))

    row_count, col_count = validate_bronze_table("psa_ds_31", "DS_31", SOURCE_FILES["DS_31"]["file"])
    display(spark.table(bronze_table("psa_ds_31")).limit(10))
    record_ingestion("DS_31", SOURCE_FILES["DS_31"]["file"], None,
        "psa_ds_31", row_count, col_count, "INGESTED")
except Exception as e:
    print(f"✗ FAILED: DS_31 — {e}")
    log_ingestion_error("DS_31", SOURCE_FILES["DS_31"]["file"], None,
        SOURCE_FILES["DS_31"]["url"], "psa_ds_31", "INGESTION_ERROR", str(e))
    record_ingestion("DS_31", SOURCE_FILES["DS_31"]["file"], None,
        "psa_ds_31", 0, 0, "FAILED", str(e))

### DS_26 to DS_29 — Inspection

These four CSV files were previously marked as NEEDS_REVIEW but are now confirmed PSA datasets:

* **DS_26** — Table 3.63: Accommodation & Food Service GVA 2022-2024
* **DS_27** — Table 3.72: Total GDP at Current Prices 2019-2023
* **DS_28** — Table 3.73: Total GDP at Constant Prices 2019-2023
* **DS_29** — Table 3.74: Per Capita GDP at Current Prices 2019-2023

Before ingesting, we inspect each raw CSV (header=false) to identify the real header position, just as we did for DS_30-32.

**Do NOT assume skipRows=1** — inspection must confirm it for each file.

In [0]:
# ── DS_26 to DS_29: Inspect raw CSV structure ──────────────────────
for ds_id in ["DS_26", "DS_27", "DS_28", "DS_29"]:
    url = SOURCE_FILES[ds_id]["url"]
    raw = (
        spark.read
        .format("csv")
        .option("databricks.connection", GOOGLE_DRIVE_CONNECTION)
        .option("header", "false")
        .load(url)
    )
    print(f"\n{'='*60}")
    print(f"{ds_id} — {SOURCE_FILES[ds_id]['file']}")
    print(f"Raw row count: {raw.count()}")
    display(raw.limit(5))

### DS_26 to DS_29 — Ingest

Inspection confirmed the same structure for all four files:
* Row 1 — Report title (e.g., "TABLE 3.63 ...")
* Row 2 — Header row (column names)
* Row 3+ — Data rows

`skipRows=1` and `header=true` are justified for all four files.

Each is ingested into its own Bronze table: `psa_ds_26`, `psa_ds_27`, `psa_ds_28`, `psa_ds_29`.

Metadata: `_source_dataset_id` = the DS ID, `_source_file` = full filename, `_source_sheet` = NULL (CSV source).

In [0]:
# ── DS_26 to DS_29: Ingest with skipRows=1 (confirmed by inspection) ──
import re

def _sanitize_csv_col(name):
    """Replace invalid Delta column name characters with underscores."""
    return re.sub(r'[^a-zA-Z0-9_]', '_', str(name).strip())

for ds_id in ["DS_26", "DS_27", "DS_28", "DS_29"]:
  try:
    url = SOURCE_FILES[ds_id]["url"]
    df = (
        spark.read
        .format("csv")
        .option("databricks.connection", GOOGLE_DRIVE_CONNECTION)
        .option("skipRows", 1)
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "FAILFAST")
        .load(url)
    )
    # Sanitize column names for Delta/UC compatibility (some headers contain spaces)
    df = df.toDF(*[_sanitize_csv_col(c) for c in df.columns])
    tbl_name = f"psa_{ds_id.lower()}"
    bronze_df = add_bronze_metadata(df, ds_id, SOURCE_FILES[ds_id]["file"],
        SOURCE_FILES[ds_id]["url"])
    (bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(bronze_table(tbl_name)))
    print(f"\n{ds_id} - {SOURCE_FILES[ds_id]['file']}")
    row_count, col_count = validate_bronze_table(tbl_name, ds_id, SOURCE_FILES[ds_id]["file"])
    display(spark.table(bronze_table(tbl_name)).limit(5))
    record_ingestion(ds_id, SOURCE_FILES[ds_id]["file"], None, tbl_name,
        row_count, col_count, "INGESTED")
  except Exception as e:
    print(f"✗ FAILED: {ds_id} — {e}")
    tbl_name = f"psa_{ds_id.lower()}"
    log_ingestion_error(ds_id, SOURCE_FILES[ds_id]["file"], None,
        SOURCE_FILES[ds_id]["url"], tbl_name, "INGESTION_ERROR", str(e))
    record_ingestion(ds_id, SOURCE_FILES[ds_id]["file"], None, tbl_name,
        0, 0, "FAILED", str(e))

## Step 4 — Excel Sources

The PSA folder contains four Excel workbooks:

* **DS_01_PSA_PTSA_Statistical_Tables_2000_2025.xlsx** — 2000–2025 Philippine Tourism Satellite Account (PTSA) Statistical Tables (10 sheets)
* **DS_02_PSA_Tourism_Statistical_Classification_2025.xlsx** — Tourism statistical classification (3 sheets)
* **DS_18_PSA_Projected_Mid_Year_Population_2020_2025.xlsx** — Projected mid-year population (1 sheet)
* **DS_08_to_DS_17_PSA_PSY_Tables_8_1_to_8_10.xlsx** — Philippine Statistical Yearbook tables (10 sheets, mapped to DS_08 through DS_17)

For each workbook we:
1. Inspect sheet names to understand workbook structure
2. Identify which sheets contain tabular data
3. Find the real header row (distinguishing report-title rows from headers)
4. Preserve original values without business cleaning
5. Ingest logically different sheets into independent Bronze tables
6. Add ingestion metadata columns

In [0]:
%pip install openpyxl

In [0]:
# ── Excel ingestion helpers ──────────────────────────────────────────
# These functions handle reading Excel workbooks from Google Drive via the
# binaryFile connector, parsing with pandas/openpyxl, converting to Spark
# DataFrames, and writing to Bronze Delta tables with metadata.

import re
import pandas as pd
import io

def sanitize_sheet_name(name: str) -> str:
    """Sanitize a worksheet name for use in a Bronze table name."""
    return re.sub(r'[^a-zA-Z0-9]', '_', name).lower()

def sanitize_column_name(name) -> str:
    """Sanitize a column name for Delta/Unity Catalog compatibility.

    Delta disallows spaces, commas, semicolons, braces, parentheses,
    newlines, tabs, and equals signs in column names.  We replace any
    character that is not alphanumeric or underscore with '_'.
    This is technically required for Delta table creation.
    """
    return re.sub(r'[^a-zA-Z0-9_]', '_', str(name).strip())

def read_excel_bytes(url: str, conn: str = GOOGLE_DRIVE_CONNECTION) -> bytes:
    """Read an Excel file from Google Drive as bytes via the binaryFile connector."""
    binary_df = (
        spark.read
        .format("binaryFile")
        .option("databricks.connection", conn)
        .load(url)
    )
    return binary_df.collect()[0]["content"]

def list_excel_sheets(url: str, conn: str = GOOGLE_DRIVE_CONNECTION):
    """List sheet names in an Excel workbook from Google Drive."""
    raw_bytes = read_excel_bytes(url, conn)
    xl = pd.ExcelFile(io.BytesIO(raw_bytes))
    return xl.sheet_names, raw_bytes

def ingest_excel_sheet(
    excel_bytes: bytes,
    sheet_name: str,
    url: str,
    dataset_id: str,
    source_file: str,
    bronze_tbl: str,
    skiprows: int = 3,
):
    """
    Read an Excel sheet with pandas, convert to Spark, add metadata,
    write to Bronze Delta table, and validate.

    All source columns are preserved as strings because Excel columns
    often contain mixed types (numbers, annotations like '2023r', NaN)
    that would be lost or cause Arrow conversion errors if forced to a
    single numeric type.  This is a technical parsing necessity for Bronze
    preservation; Silver will handle type conversion.
    """
    # Read with pandas (header at the specified row)
    pdf = pd.read_excel(io.BytesIO(excel_bytes), sheet_name=sheet_name, header=skiprows)

    # Drop completely empty rows
    pdf = pdf.dropna(how='all')

    # Sanitize column names (years come as numbers, and some header cells
    # contain spaces/invalid characters for Delta)
    new_cols = []
    for i, col in enumerate(pdf.columns):
        if col is None or (isinstance(col, float) and pd.isna(col)):
            new_cols.append(f"Unnamed_{i}")
        else:
            new_cols.append(sanitize_column_name(col))
    # Handle duplicate column names by appending a suffix
    seen = {}
    for i, c in enumerate(new_cols):
        if c in seen:
            seen[c] += 1
            new_cols[i] = f"{c}_{seen[c]}"
        else:
            seen[c] = 0
    pdf.columns = new_cols

    # Convert all columns to string for Bronze preservation
    for col in pdf.columns:
        pdf[col] = pdf[col].astype(str)
    pdf = pdf.replace({'nan': None, 'NaT': None, 'None': None})

    # Convert to Spark DataFrame
    df = spark.createDataFrame(pdf)

    # Add metadata columns
    df = add_bronze_metadata(df, dataset_id, source_file, url, source_sheet=sheet_name)

    # Write to Bronze (overwrite for idempotency)
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(bronze_table(bronze_tbl)))

    # Validate
    table_df = spark.table(bronze_table(bronze_tbl))
    row_count = table_df.count()
    col_count = len(table_df.columns)

    for col in METADATA_COLS:
        assert col in table_df.columns, f"Missing metadata column: {col}"
    # Verify metadata values
    assert table_df.filter(F.col("_source_system") != "PSA").count() == 0
    assert table_df.filter(F.col("_source_path").isNull()).count() == 0
    assert table_df.filter(F.col("_ingested_at").isNull()).count() == 0
    assert table_df.filter(F.col("_source_dataset_id") != dataset_id).count() == 0
    assert table_df.filter(F.col("_source_file") != source_file).count() == 0
    assert table_df.filter(F.col("_source_sheet").isNull()).count() == 0

    print(f"  + {bronze_tbl} | Rows: {row_count} | Cols: {col_count} | Sheet: {sheet_name}")
    record_ingestion(dataset_id, source_file, sheet_name, bronze_tbl, row_count, col_count, "INGESTED", f"Sheet: {sheet_name}", source_path=url)
    return row_count, col_count

print("Excel ingestion helpers loaded.")

### DS_01.xlsx — Inspect Workbook Structure

**Source file:** DS_01_PSA_PTSA_Statistical_Tables_2000_2025.xlsx
**Bronze target:** `tuklas_dev`.`02_bronze`.`psa_ds_01_table_01` through `psa_ds_01_table_10` (one per sheet)

First we list the worksheet names in the workbook to understand its structure.

In [0]:
# ── DS_01: Inspect workbook structure ───────────────────────────────
ds01_url = SOURCE_FILES["DS_01"]["url"]

# Read Excel file as binary through the Google Drive connection, then parse with pandas.
import pandas as pd
import io

ds01_binary = (
    spark.read
    .format("binaryFile")
    .option("databricks.connection", GOOGLE_DRIVE_CONNECTION)
    .load(ds01_url)
)

# Collect binary content (file is small enough for driver memory)
ds01_row = ds01_binary.collect()[0]
ds01_bytes = ds01_row["content"]
print(f"DS_01 binary size: {len(ds01_bytes)} bytes")

# Parse with pandas/openpyxl to discover sheet names
xl = pd.ExcelFile(io.BytesIO(ds01_bytes))
print(f"DS_01 sheet names: {xl.sheet_names}")

# Display first 10 rows of each sheet (raw, no header assumption)
for sheet in xl.sheet_names:
    print(f"\n{'='*60}")
    print(f"Sheet: {sheet}")
    raw = pd.read_excel(io.BytesIO(ds01_bytes), sheet_name=sheet, header=None, nrows=10)
    print(f"Shape: {raw.shape}")
    print(raw.to_string())

### DS_01.xlsx — Ingest All Sheets

Inspection confirmed that all 10 sheets in DS_01 follow the same structure:

* Row 0 — Table title
* Row 1 — Subtitle (e.g., "Levels (in million PhP)")
* Row 2 — Blank
* Row 3 — Header row (product name + year columns 2000–2025)
* Row 4+ — Data rows

Each sheet is ingested into its own Bronze table using an explicit mapping: `psa_ds_01_table_01` through `psa_ds_01_table_10`.

In [0]:
# ── DS_01: Ingest all 10 sheets ──────────────────────────────────────
# ds01_bytes and xl.sheet_names are available from the inspection cell.
# All sheets use skiprows=3 (header in row 3).

ds01_sheets = xl.sheet_names
print(f"Ingesting DS_01: {len(ds01_sheets)} sheets\n")

ds01_file = SOURCE_FILES["DS_01"]["file"]
for sheet in ds01_sheets:
    tbl_name = DS01_TABLE_MAP[sheet]
    try:
        ingest_excel_sheet(ds01_bytes, sheet, ds01_url, "DS_01",
            ds01_file, tbl_name, skiprows=3)
    except Exception as e:
        print(f"✗ FAILED: DS_01/{sheet} -> {tbl_name} — {e}")
        log_ingestion_error("DS_01", ds01_file, sheet, ds01_url, tbl_name,
            "INGESTION_ERROR", str(e))
        record_ingestion("DS_01", ds01_file, sheet, tbl_name,
            0, 0, "FAILED", str(e))

print(f"\nDS_01 ingestion complete: {len(ds01_sheets)} sheets processed.")

### DS_02.xlsx — Inspect Workbook Structure

**Source file:** DS_02_PSA_Tourism_Statistical_Classification_2025.xlsx
**Bronze target:** `tuklas_dev`.`02_bronze`.`psa_ds_02_metadata`, `psa_ds_02_tourism_industries`, `psa_ds_02_tourism_products` (one per sheet)

We list sheet names and inspect the first rows of each sheet to identify the header position.

In [0]:
# ── DS_02: Inspect workbook structure ───────────────────────────────
ds02_url = SOURCE_FILES["DS_02"]["url"]

ds02_sheet_names, ds02_bytes = list_excel_sheets(ds02_url)
print(f"DS_02 sheet names: {ds02_sheet_names}")
print(f"DS_02 binary size: {len(ds02_bytes)} bytes")

for sheet in ds02_sheet_names:
    print(f"\n{'='*60}")
    print(f"Sheet: {sheet}")
    raw = pd.read_excel(io.BytesIO(ds02_bytes), sheet_name=sheet, header=None, nrows=10)
    print(f"Shape: {raw.shape}")
    print(raw.to_string())

### DS_18 (Excel) — Inspect Workbook Structure

**Source file:** DS_18_PSA_Projected_Mid_Year_Population_2020_2025.xlsx
**Bronze target:** `tuklas_dev`.`02_bronze`.`psa_ds_18` (or per-sheet tables)

Note: A PDF version of DS_18 also exists and will be handled separately in Step 5.

In [0]:
# ── DS_18 (Excel): Inspect workbook structure ────────────────────────
ds18_xlsx_url = SOURCE_FILES["DS_18_xlsx"]["url"]

ds18_sheet_names, ds18_bytes = list_excel_sheets(ds18_xlsx_url)
print(f"DS_18 sheet names: {ds18_sheet_names}")
print(f"DS_18 binary size: {len(ds18_bytes)} bytes")

for sheet in ds18_sheet_names:
    print(f"\n{'='*60}")
    print(f"Sheet: {sheet}")
    raw = pd.read_excel(io.BytesIO(ds18_bytes), sheet_name=sheet, header=None, nrows=10)
    print(f"Shape: {raw.shape}")
    print(raw.to_string())

### PSY Tables — Inspect Workbook Structure

**Source file:** DS_08_to_DS_17_PSA_PSY_Tables_8_1_to_8_10.xlsx
**Bronze target:** `tuklas_dev`.`02_bronze`.`psa_ds_08` through `psa_ds_17` (one per sheet)

PSY = Philippine Statistical Yearbook. This single physical workbook contains 10 sheets (T8_1 through T8_10), each mapped to a canonical DS identifier: T8_1 → DS_08, T8_2 → DS_09, ..., T8_10 → DS_17.

In [0]:
# ── PSY Tables: Inspect workbook structure ───────────────────────────
psy_url = SOURCE_FILES["DS_08_17"]["url"]

psy_sheet_names, psy_bytes = list_excel_sheets(psy_url)
print(f"PSY Tables sheet names: {psy_sheet_names}")
print(f"PSY Tables binary size: {len(psy_bytes)} bytes")

for sheet in psy_sheet_names:
    print(f"\n{'='*60}")
    print(f"Sheet: {sheet}")
    raw = pd.read_excel(io.BytesIO(psy_bytes), sheet_name=sheet, header=None, nrows=10)
    print(f"Shape: {raw.shape}")
    print(raw.to_string())

### DS_02.xlsx — Ingest All Sheets

DS_02 has 3 sheets with different structures:

* **Metadata** — Key-value pairs (no tabular header). Ingested with no header.
* **2025 Tourism Industries** — Header in row 2 (title in row 0, blank row 1). `skiprows=2`.
* **2025 Tourism Products** — Header in row 2 (same structure). `skiprows=2`.

In [0]:
# ── DS_02: Ingest all 3 sheets ──────────────────────────────────────
print(f"Ingesting DS_02: {len(ds02_sheet_names)} sheets\n")

for sheet in ds02_sheet_names:
  try:
    tbl_name = DS02_TABLE_MAP[sheet]
    if sheet == "Metadata":
        # Metadata sheet: key-value pairs, no tabular header
        pdf = pd.read_excel(io.BytesIO(ds02_bytes), sheet_name=sheet, header=None)
        pdf = pdf.dropna(how='all')
        pdf.columns = [f"_{i}" for i in range(len(pdf.columns))]
        for col in pdf.columns:
            pdf[col] = pdf[col].astype(str)
        pdf = pdf.replace({'nan': None, 'NaT': None, 'None': None})
        df = spark.createDataFrame(pdf)
        df = add_bronze_metadata(df, "DS_02", SOURCE_FILES["DS_02"]["file"], ds02_url, source_sheet=sheet)
        (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
            .saveAsTable(bronze_table(tbl_name)))
        table_df = spark.table(bronze_table(tbl_name))
        row_count = table_df.count()
        col_count = len(table_df.columns)
        for col in METADATA_COLS:
            assert col in table_df.columns, f"Missing metadata column: {col}"
        assert table_df.filter(F.col("_source_system") != "PSA").count() == 0
        assert table_df.filter(F.col("_source_path").isNull()).count() == 0
        assert table_df.filter(F.col("_ingested_at").isNull()).count() == 0
        assert table_df.filter(F.col("_source_dataset_id") != "DS_02").count() == 0
        assert table_df.filter(F.col("_source_file") != SOURCE_FILES["DS_02"]["file"]).count() == 0
        assert table_df.filter(F.col("_source_sheet").isNull()).count() == 0
        print(f"  + {tbl_name} | Rows: {row_count} | Cols: {col_count} | Sheet: {sheet}")
        record_ingestion("DS_02", SOURCE_FILES["DS_02"]["file"], sheet, tbl_name, row_count, col_count, "INGESTED", f"Sheet: {sheet} (metadata, no header)")
    else:
        # Tabular sheets: header in row 2
        ingest_excel_sheet(ds02_bytes, sheet, ds02_url, "DS_02",
            SOURCE_FILES["DS_02"]["file"], tbl_name, skiprows=2)
  except Exception as e:
    print(f"✗ FAILED: DS_02/{sheet} -> {tbl_name} — {e}")
    log_ingestion_error("DS_02", SOURCE_FILES["DS_02"]["file"], sheet, ds02_url, tbl_name,
        "INGESTION_ERROR", str(e))
    record_ingestion("DS_02", SOURCE_FILES["DS_02"]["file"], sheet, tbl_name,
        0, 0, "FAILED", str(e))

print(f"\nDS_02 ingestion complete.")

### DS_18 (Excel) — Ingest

DS_18 has a single sheet with a complex 3-level merged header (section headers, years, Total/Male/Female sub-headers). For Bronze preservation, we skip the title and subtitle rows (rows 0-1) and read the rest with no header, preserving the multi-level header structure as data rows. The Silver layer will handle proper header parsing.

**Bronze target:** `tuklas_dev`.`02_bronze`.`psa_ds_18`

In [0]:
# ── DS_18: Ingest single sheet (multi-level header preserved as data) ──
print(f"Ingesting DS_18: {len(ds18_sheet_names)} sheets\n")

for sheet in ds18_sheet_names:
  try:
    tbl_name = "psa_ds_18"
    # Skip title+subtitle (rows 0-1), read rest with no header
    pdf = pd.read_excel(io.BytesIO(ds18_bytes), sheet_name=sheet, header=None, skiprows=2)
    pdf = pdf.dropna(how='all')
    pdf.columns = [f"_{i}" for i in range(len(pdf.columns))]
    for col in pdf.columns:
        pdf[col] = pdf[col].astype(str)
    pdf = pdf.replace({'nan': None, 'NaT': None, 'None': None})
    df = spark.createDataFrame(pdf)
    df = add_bronze_metadata(df, "DS_18", SOURCE_FILES["DS_18_xlsx"]["file"], ds18_xlsx_url, source_sheet=sheet)
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(bronze_table(tbl_name)))
    table_df = spark.table(bronze_table(tbl_name))
    row_count = table_df.count()
    col_count = len(table_df.columns)
    for col in METADATA_COLS:
        assert col in table_df.columns, f"Missing metadata column: {col}"
    assert table_df.filter(F.col("_source_system") != "PSA").count() == 0
    assert table_df.filter(F.col("_source_path").isNull()).count() == 0
    assert table_df.filter(F.col("_ingested_at").isNull()).count() == 0
    assert table_df.filter(F.col("_source_dataset_id") != "DS_18").count() == 0
    assert table_df.filter(F.col("_source_file") != SOURCE_FILES["DS_18_xlsx"]["file"]).count() == 0
    assert table_df.filter(F.col("_source_sheet").isNull()).count() == 0
    print(f"  + {tbl_name} | Rows: {row_count} | Cols: {col_count} | Sheet: {sheet}")
    record_ingestion("DS_18", SOURCE_FILES["DS_18_xlsx"]["file"], sheet, tbl_name, row_count, col_count, "INGESTED", f"Sheet: {sheet} (multi-level header preserved as data)")
  except Exception as e:
    print(f"✗ FAILED: DS_18/{sheet} -> {tbl_name} — {e}")
    log_ingestion_error("DS_18", SOURCE_FILES["DS_18_xlsx"]["file"], sheet, ds18_xlsx_url, tbl_name,
        "INGESTION_ERROR", str(e))
    record_ingestion("DS_18", SOURCE_FILES["DS_18_xlsx"]["file"], sheet, tbl_name,
        0, 0, "FAILED", str(e))

print(f"\nDS_18 ingestion complete.")

### PSY Tables — Ingest All Sheets

PSY Tables has 10 sheets (T8_1 through T8_10). Each sheet has:

* Row 0 — Table title
* Row 1 — Blank
* Row 2 — Header row
* Row 3+ — Data rows

Each sheet is ingested into its own Bronze table: `psa_ds_08` through `psa_ds_17`, following the mapping T8_1→DS_08, T8_2→DS_09, ..., T8_10→DS_17.

In [0]:
# ── PSY Tables: Ingest all 10 sheets (mapped to DS_08–DS_17) ───────────
psy_file = SOURCE_FILES["DS_08_17"]["file"]
print(f"Ingesting PSY Tables: {len(psy_sheet_names)} sheets -> DS_08 to DS_17\n")

for sheet in psy_sheet_names:
    ds_id = PSY_SHEET_TO_DS[sheet]
    tbl_name = f"psa_{ds_id.lower()}"
    try:
        ingest_excel_sheet(psy_bytes, sheet, psy_url, ds_id,
            psy_file, tbl_name, skiprows=2)
    except Exception as e:
        print(f"✗ FAILED: {ds_id}/{sheet} -> {tbl_name} — {e}")
        log_ingestion_error(ds_id, psy_file, sheet, psy_url, tbl_name,
            "INGESTION_ERROR", str(e))
        record_ingestion(ds_id, psy_file, sheet, tbl_name,
            0, 0, "FAILED", str(e))

print(f"\nPSY Tables ingestion complete: {len(psy_sheet_names)} sheets processed.")

## Step 5 — PDF Source

A PDF version of DS_18 also exists in the Google Drive folder. This PDF is **not** treated as the same structured table as the Excel file.

We attempt to ingest it as binary/unstructured Bronze data in `tuklas_dev`.`02_bronze`.`psa_ds_18_pdf`. If the binary ingestion is not feasible, we document that the PDF was intentionally excluded from structured Bronze ingestion because the Excel source (`psa_ds_18`) provides the tabular representation.

No PDF-to-table extraction or analytical cleanup is performed in this notebook.

In [0]:
# ── DS_18 PDF: Attempt binary ingestion to Bronze ──────────────────────
ds18_pdf_url = SOURCE_FILES["DS_18_pdf"]["url"]

try:
    pdf_binary = (
        spark.read
        .format("binaryFile")
        .option("databricks.connection", GOOGLE_DRIVE_CONNECTION)
        .load(ds18_pdf_url)
    )
    pdf_row = pdf_binary.collect()[0]
    pdf_bytes = pdf_row["content"]
    print(f"DS_18 PDF binary size: {len(pdf_bytes)} bytes")

    # Write as binary Bronze table (single row with binary content + metadata)
    pdf_df = pdf_binary.select(
        F.lit("DS_18").alias("_source_dataset_id"),
        F.lit(SOURCE_FILES["DS_18_pdf"]["file"]).alias("_source_file"),
        F.lit(None).cast("string").alias("_source_sheet"),
        F.lit(ds18_pdf_url).alias("_source_path"),
        F.lit("PSA").alias("_source_system"),
        F.current_timestamp().alias("_ingested_at"),
        F.col("path"),
        F.col("length").alias("file_size"),
        F.col("content").alias("binary_content"),
    )

    (pdf_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(bronze_table("psa_ds_18_pdf")))

    table_df = spark.table(bronze_table("psa_ds_18_pdf"))
    row_count = table_df.count()
    col_count = len(table_df.columns)

    for col in METADATA_COLS:
        assert col in table_df.columns, f"Missing metadata column: {col}"
    assert table_df.filter(F.col("_source_system") != "PSA").count() == 0
    assert table_df.filter(F.col("_source_path").isNull()).count() == 0
    assert table_df.filter(F.col("_ingested_at").isNull()).count() == 0
    assert table_df.filter(F.col("_source_dataset_id") != "DS_18").count() == 0
    assert table_df.filter(F.col("_source_file") != SOURCE_FILES["DS_18_pdf"]["file"]).count() == 0
    assert table_df.filter(F.col("_source_sheet").isNotNull()).count() == 0

    print(f"Table  : {bronze_table('psa_ds_18_pdf')}")
    print(f"Rows   : {row_count}")
    print(f"Columns: {col_count}")
    print("Binary PDF ingested as unstructured Bronze data.")
    record_ingestion("DS_18", SOURCE_FILES["DS_18_pdf"]["file"], None, "psa_ds_18_pdf", row_count, col_count, "INGESTED", "Binary/unstructured Bronze data")

except Exception as e:
    print(f"PDF binary ingestion failed: {e}")
    print("DS_18 PDF was intentionally excluded from STRUCTURED Bronze ingestion.")
    print("The Excel source (psa_ds_18) provides the tabular representation.")
    log_ingestion_error("DS_18", SOURCE_FILES["DS_18_pdf"]["file"], None,
        SOURCE_FILES["DS_18_pdf"]["url"], "psa_ds_18_pdf", "PDF_INGESTION_ERROR", str(e))
    record_ingestion("DS_18", SOURCE_FILES["DS_18_pdf"]["file"], None, "psa_ds_18_pdf", 0, 0, "FAILED", f"PDF ingestion failed: {e}")

## Cleanup — Drop Obsolete Bronze Tables

The previous version of this notebook created Bronze tables with names that have since been standardized:

* `psa_psy_t8_1` through `psa_psy_t8_10` — replaced by `psa_ds_08` through `psa_ds_17`
* `psa_ds_01_tables_1` through `psa_ds_01_tables_10` — replaced by `psa_ds_01_table_01` through `psa_ds_01_table_10`
* `psa_ds_02_2025_tourism_industries` and `psa_ds_02_2025_tourism_products` — replaced by `psa_ds_02_tourism_industries` and `psa_ds_02_tourism_products`

After verifying the replacement tables exist, we drop the obsolete tables.
Only tables created by this notebook are dropped — unrelated tables are left untouched.

In [0]:
# ── Cleanup: Drop obsolete psa_psy_t8_* Bronze tables ──────────────
# These were created by the previous version of this notebook and are
# now replaced by psa_ds_08 through psa_ds_17.
old_psy_tables = [f"psa_psy_t8_{i}" for i in range(1, 11)]
for tbl in old_psy_tables:
    full_name = bronze_table(tbl)
    if spark.catalog.tableExists(full_name):
        print(f"Dropping obsolete table: {full_name}")
        spark.sql(f"DROP TABLE IF EXISTS {full_name}")
    else:
        print(f"Already absent: {full_name}")
print("\nCleanup complete.")

# ── Drop obsolete DS_01 tables (old naming) ──────────────────────────
old_ds01_tables = [
    "psa_ds_01_tables_1", "psa_ds_01_tables_2", "psa_ds_01_tables_3",
    "psa_ds_01_tables_4", "psa_ds_01_tables_5", "psa_ds_01_tables_6",
    "psa_ds_01_table_7", "psa_ds_01_tables_8", "psa_ds_01_tables_9",
    "psa_ds_01_tables_10",
]
for tbl in old_ds01_tables:
    full_name = bronze_table(tbl)
    if spark.catalog.tableExists(full_name):
        print(f"Dropping obsolete table: {full_name}")
        spark.sql(f"DROP TABLE IF EXISTS {full_name}")
    else:
        print(f"Already absent: {full_name}")

# ── Drop obsolete DS_02 tables (old naming) ──────────────────────────
old_ds02_tables = ["psa_ds_02_2025_tourism_industries", "psa_ds_02_2025_tourism_products"]
for tbl in old_ds02_tables:
    full_name = bronze_table(tbl)
    if spark.catalog.tableExists(full_name):
        print(f"Dropping obsolete table: {full_name}")
        spark.sql(f"DROP TABLE IF EXISTS {full_name}")
    else:
        print(f"Already absent: {full_name}")

print("\nDS_01/DS_02 cleanup complete.")

## Source Quality Notes

The following data quality artifacts were observed in the PSA source files. These are **preserved unchanged** in the Bronze layer and should be corrected later in Silver using authoritative PSGC mappings.

### Unicode Replacement Character

Some DS_32, DS_30, and DS_31 geographic values contain the Unicode replacement character (\ufffd), including values similar to:

* `Las Pi\ufffdas` (should be Las Piñas)
* `Para\ufffdaque` (should be Parañaque)

This artifact exists in the source CSV files and must remain unchanged in Bronze. Correction should happen in Silver.

### Leading Dots Hierarchy Markers

Some rows contain leading dots that appear to encode geographic hierarchy:

* `..National Capital Region` (DS_30) — 2-dot marker for region
* `....City of Caloocan` (DS_30) — 4-dot marker for city under region
* `..City of Caloocan` (DS_31, DS_32) — 2-dot marker for city

These markers are preserved exactly because they may represent source hierarchy levels. Hierarchy parsing belongs in Silver.

### Excel Mixed-Type Columns

Excel workbook columns often contain mixed types (numbers, text annotations like `2023r` for "revised", and blank cells). All Excel source columns are preserved as strings in Bronze to avoid data loss from type coercion. Silver will handle proper type conversion.

# Bronze Ingestion Summary

The table below summarizes every file discovered in the PSA Google Drive folder and its Bronze ingestion status.

Allowed statuses: `INGESTED`, `SKIPPED`, `NEEDS_REVIEW`, `FAILED`.

In [0]:
# ── Build and display the ingestion summary ───────────────────────────
# Deduplicate by (source_dataset_id, bronze_table), keeping the
# latest entry (most recent run result wins).
_seen = {}
for _r in ingestion_summary:
    _dsid = _r.get("source_dataset_id", "")
    if not _dsid:
        continue  # Skip old entries without source_dataset_id
    _bt = _r.get("bronze_table", "")
    _key = (_dsid, _bt)
    _seen[_key] = _r
_clean_summary = list(_seen.values())

# ── Reconcile Drive folder listing against SOURCE_FILES ────────────
# Compare file IDs from path column (folder_listing paths don't have /view suffix)
import re

def _extract_file_id(url):
    _m = re.search(r'/file/d/([^/]+)', url)
    return _m.group(1) if _m else url

drive_ids = set()
_path_lookup = {}
for _row in folder_listing.select("path").collect():
    _fid = _extract_file_id(_row["path"])
    drive_ids.add(_fid)
    _path_lookup[_fid] = _row["path"]
known_ids = set(_extract_file_id(v["url"]) for v in SOURCE_FILES.values())
for _fid in sorted(drive_ids - known_ids):
    _path = _path_lookup.get(_fid, _fid)
    _clean_summary.append({
        "source_dataset_id": "UNKNOWN", "source_file": _path,
        "source_sheet": None, "bronze_table": "N/A",
        "row_count": 0, "column_count": 0,
        "status": "NEEDS_REVIEW",
        "notes": "Discovered in Drive folder but not in SOURCE_FILES",
    })
    print(f"  NEEDS_REVIEW: {_path}")

summary_df = spark.createDataFrame(_clean_summary)

print(f"Total records in summary: {summary_df.count()}")
print()
display(summary_df.orderBy("source_dataset_id"))

print("\nStatus counts:")
summary_df.groupBy("status").count().orderBy(F.col("count").desc()).show()

# ── Update ingestion_run_log from STARTED to final status ─────────
# Roll up manifest rows by physical source_file to get file-level metrics.
# One physical file may produce multiple logical outputs (e.g., DS_01 -> 10 tables).
# File-level precedence: FAILED > INGESTED > SKIPPED/NEEDS_REVIEW
manifest_df = spark.table(ingestion_table("file_manifest")).filter(F.col("run_id") == run_id)
manifest_df.createOrReplaceTempView("_run_manifest")

# One row per physical source_file with rolled-up status
spark.sql("""
SELECT source_file,
    CASE
        WHEN SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) > 0 THEN 'FAILED'
        WHEN SUM(CASE WHEN status = 'INGESTED' THEN 1 ELSE 0 END) > 0 THEN 'INGESTED'
        WHEN SUM(CASE WHEN status IN ('SKIPPED','NEEDS_REVIEW') THEN 1 ELSE 0 END) > 0 THEN 'SKIPPED'
        ELSE 'UNKNOWN'
    END AS file_status
FROM _run_manifest
GROUP BY source_file
""").createOrReplaceTempView("_file_rollup")

# File-level metrics (unique physical source files)
files_discovered = spark.sql("SELECT COUNT(*) FROM _file_rollup").collect()[0][0]
files_ingested   = spark.sql("SELECT COUNT(*) FROM _file_rollup WHERE file_status = 'INGESTED'").collect()[0][0]
files_skipped    = spark.sql("SELECT COUNT(*) FROM _file_rollup WHERE file_status = 'SKIPPED'").collect()[0][0]
files_failed     = spark.sql("SELECT COUNT(*) FROM _file_rollup WHERE file_status = 'FAILED'").collect()[0][0]

# Tables created = distinct successfully ingested destination_table values
tables_created = spark.sql(
    "SELECT COUNT(DISTINCT destination_table) FROM _run_manifest WHERE status = 'INGESTED'"
).collect()[0][0]

errors_df = spark.table(ingestion_table("ingestion_errors")).filter(F.col("run_id") == run_id)
error_count = errors_df.count()

# Determine final status based on file-level rollup
if files_failed == 0 and error_count == 0:
    final_status = "SUCCESS"
elif files_ingested > 0:
    final_status = "PARTIAL_SUCCESS"
else:
    final_status = "FAILED"

# Generate dynamic notes (do not hardcode numbers)
if final_status == "SUCCESS":
    final_notes = (
        f"PSA ingestion completed successfully: {files_discovered} physical files processed, "
        f"{tables_created} Bronze outputs ingested, {files_failed} failed, "
        f"{files_skipped} unresolved, {error_count} errors."
    )
else:
    final_notes = (
        f"PSA ingestion completed with issues: {files_discovered} physical files processed, "
        f"{files_ingested} ingested, {files_failed} failed, {files_skipped} unresolved, "
        f"{tables_created} Bronze outputs, {error_count} errors."
    )

# Update the SAME ingestion_run_log row (STARTED -> final status)
spark.sql(f"""
UPDATE {ingestion_table('ingestion_run_log')}
SET status = '{final_status}',
    completed_at = current_timestamp(),
    files_discovered = {files_discovered},
    files_ingested = {files_ingested},
    files_skipped = {files_skipped},
    files_failed = {files_failed},
    tables_created = {tables_created},
    notes = '{final_notes}'
WHERE run_id = '{run_id}'
""")

# ── Reconcile stale STARTED runs ──────────────────────────────────
# Find prior run_ids still STARTED (completed_at NULL) that already have
# manifest records.  Either reconcile or mark as interrupted -- never delete.
stale_runs = spark.sql(f"""
SELECT r.run_id,
       (SELECT COUNT(*) FROM {ingestion_table('file_manifest')} m
        WHERE m.run_id = r.run_id) AS manifest_count
FROM {ingestion_table('ingestion_run_log')} r
WHERE r.status = 'STARTED'
  AND r.completed_at IS NULL
  AND r.run_id != '{run_id}'
  AND EXISTS (SELECT 1 FROM {ingestion_table('file_manifest')} m
             WHERE m.run_id = r.run_id)
""").collect()

for stale in stale_runs:
    stale_run_id = stale["run_id"]
    spark.table(ingestion_table("file_manifest")).filter(
        F.col("run_id") == stale_run_id
    ).createOrReplaceTempView("_stale_manifest")

    spark.sql("""
    SELECT source_file,
        CASE
            WHEN SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) > 0 THEN 'FAILED'
            WHEN SUM(CASE WHEN status = 'INGESTED' THEN 1 ELSE 0 END) > 0 THEN 'INGESTED'
            WHEN SUM(CASE WHEN status IN ('SKIPPED','NEEDS_REVIEW') THEN 1 ELSE 0 END) > 0 THEN 'SKIPPED'
            ELSE 'UNKNOWN'
        END AS file_status
    FROM _stale_manifest GROUP BY source_file
    """).createOrReplaceTempView("_stale_rollup")

    s_fd = spark.sql("SELECT COUNT(*) FROM _stale_rollup").collect()[0][0]
    s_fi = spark.sql("SELECT COUNT(*) FROM _stale_rollup WHERE file_status='INGESTED'").collect()[0][0]
    s_fs = spark.sql("SELECT COUNT(*) FROM _stale_rollup WHERE file_status='SKIPPED'").collect()[0][0]
    s_ff = spark.sql("SELECT COUNT(*) FROM _stale_rollup WHERE file_status='FAILED'").collect()[0][0]
    s_tc = spark.sql("SELECT COUNT(DISTINCT destination_table) FROM _stale_manifest WHERE status='INGESTED'").collect()[0][0]

    if s_ff == 0 and s_fi > 0 and s_fs == 0:
        # Manifest shows a complete successful run -> reconcile
        s_notes = f"Reconciled from stale STARTED: {s_fi} files ingested, {s_tc} Bronze outputs, 0 failed."
        spark.sql(f"""
        UPDATE {ingestion_table('ingestion_run_log')}
        SET status = 'SUCCESS',
            completed_at = (SELECT MAX(ingested_at) FROM {ingestion_table('file_manifest')}
                           WHERE run_id = '{stale_run_id}'),
            files_discovered = {s_fd},
            files_ingested = {s_fi},
            files_skipped = {s_fs},
            files_failed = {s_ff},
            tables_created = {s_tc},
            notes = '{s_notes}'
        WHERE run_id = '{stale_run_id}'
        """)
        print(f"  Reconciled stale run {stale_run_id} -> SUCCESS")
    else:
        # Manifest is incomplete or has failures -> mark INTERRUPTED
        s_notes = f"Run did not reach finalization. Reconciled: {s_fi} ingested, {s_ff} failed, {s_tc} Bronze outputs."
        spark.sql(f"""
        UPDATE {ingestion_table('ingestion_run_log')}
        SET status = 'FAILED',
            completed_at = COALESCE(
                (SELECT MAX(ingested_at) FROM {ingestion_table('file_manifest')}
                 WHERE run_id = '{stale_run_id}'),
                started_at
            ),
            files_discovered = {s_fd},
            files_ingested = {s_fi},
            files_skipped = {s_fs},
            files_failed = {s_ff},
            tables_created = {s_tc},
            notes = '{s_notes}'
        WHERE run_id = '{stale_run_id}'
        """)
        print(f"  Marked stale run {stale_run_id} -> FAILED (interrupted)")

# ── Display control-table results ─────────────────────────────────
print(f"\n{'='*60}")
print(f"  Run ID : {run_id}")
print(f"  Status : {final_status}")
print(f"  Files  : discovered={files_discovered}  ingested={files_ingested}  skipped={files_skipped}  failed={files_failed}")
print(f"  Tables : {tables_created}")
print(f"  Errors : {error_count}")
print(f"  Notes  : {final_notes}")
print(f"{'='*60}")

print(f"\n--- ingestion_run_log (current run) ---")
display(spark.table(ingestion_table("ingestion_run_log")).filter(F.col("run_id") == run_id))

print(f"\n--- file_manifest (current run) ---")
display(manifest_df.orderBy("source_dataset_id", "destination_table"))

print(f"\n--- ingestion_errors (current run) ---")
if error_count > 0:
    display(errors_df.orderBy("error_timestamp"))
else:
    print("No errors recorded for this run.")